## 17B. Multi-threshold ensemble: Champion + Sniper filter

**Цель:** проверить, даёт ли ансамбль «чемпион + снайпер» прирост при комбинировании:
- **Чемпион:** cat_seq_new prod_hold 25-70 (много сделок, ~1.95% avg)
- **Снайперы:** модели с узким порогом 20-80 (мало сделок, высокая avg: cat 2.43%, logreg 4.84%, rf 2.79%)

**Стратегия фильтра:** торговать только когда чемпион И снайпер согласны (оба BUY или оба SELL).

Split 7/1/2, данные и фичи как в NB15/NB17 (sequence_5_15_30_60).

In [1]:
import sys, os, numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

HAS_CAT = False
try:
    import catboost as cb
    HAS_CAT = True
except ImportError:
    pass

_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) in ('01_data_prep','02_targets','03_features','04_models','05_experiments') else os.getcwd()
if _root not in sys.path:
    sys.path.insert(0, _root)

COMMISSION_RT = 0.001
TARGET_COL = 'target'
print('Root:', _root, '| CAT:', HAS_CAT)

Root: c:\project\trading_bot_2Engine | CAT: True


In [2]:
# 1) Загрузка данных, rolling features (sequence_5_15_30_60) — как в NB15

labeled_path = os.path.join(_root, 'outputs', 'data_labeled_tp_sl_1_05.parquet')
feat_path = os.path.join(_root, 'outputs', 'features_selected_tp_sl_1_05.txt')
df = pd.read_parquet(labeled_path)
sort_col = 'datetime' if 'datetime' in df.columns else 'timestamp'
with open(feat_path, encoding='utf-8') as f:
    BASE_FEATURES = [l.strip() for l in f if l.strip()]
BASE_FEATURES = [c for c in BASE_FEATURES if c in df.columns]

# valid_full для rolling (BUY+SELL+HOLD) — как в NB17
valid_full = df.dropna(subset=BASE_FEATURES + [TARGET_COL]).copy()
valid_full = valid_full[valid_full[TARGET_COL].isin([-1.0, 0.0, 1.0])]
valid_full['date'] = pd.to_datetime(valid_full['datetime'], utc=True).dt.date
valid_full['ret_next'] = valid_full.groupby('session_key')['close_price'].pct_change().shift(-1)
valid_full = valid_full.dropna(subset=['ret_next'])

# Rolling на полном df
key_feats = BASE_FEATURES[:10]
df_feat = valid_full.copy()
grp = df_feat.groupby('session_key', group_keys=False)
for w in [5, 15, 30, 60]:
    for c in key_feats:
        if c in df_feat.columns:
            df_feat[f'{c}_roll{w}_mean'] = grp[c].transform(lambda x: x.rolling(w, min_periods=1).mean())
            df_feat[f'{c}_roll{w}_std'] = grp[c].transform(lambda x: x.rolling(w, min_periods=1).std().fillna(0))

new_cols = [c for c in df_feat.columns if ('_roll5_' in c or '_roll15_' in c or '_roll30_' in c or '_roll60_' in c)]
FEATURES_NEW = [c for c in (BASE_FEATURES + new_cols) if c in df_feat.columns]

dates = sorted(df_feat['date'].unique())
assert len(dates) >= 10, f'Нужно >= 10 дней, найдено {len(dates)}'
train_dates = set(dates[:7])
val_dates = set([dates[7]])
test_dates = set(dates[8:10])
val_day, test1_day, test2_day = dates[7], dates[8], dates[9]

# Split для fit (BUY+SELL)
valid = df_feat[df_feat[TARGET_COL].isin([-1.0, 1.0])].copy()
valid['y'] = (valid[TARGET_COL] == 1).astype(int)
tr = valid[valid['date'].isin(train_dates)].sort_values(['session_key', sort_col]).reset_index(drop=True)
va = valid[valid['date'].isin(val_dates)].sort_values(['session_key', sort_col]).reset_index(drop=True)
te = valid[valid['date'].isin(test_dates)].sort_values(['session_key', sort_col]).reset_index(drop=True)

# Полный тест по дням (BUY+SELL+HOLD)
te_full = df_feat[df_feat['date'].isin(test_dates)].dropna(subset=FEATURES_NEW + [TARGET_COL]).sort_values(['session_key', sort_col]).reset_index(drop=True)
val_full = df_feat[df_feat['date'] == val_day].dropna(subset=FEATURES_NEW + [TARGET_COL]).sort_values(['session_key', sort_col]).reset_index(drop=True)

print('Features:', len(FEATURES_NEW), '| train:', len(tr), 'val:', len(va), 'te:', len(te))
print('te_full (BUY+SELL+HOLD):', len(te_full), '| val_full:', len(val_full))

Features: 101 | train: 123971 val: 23108 te: 39475
te_full (BUY+SELL+HOLD): 74437 | val_full: 45592


In [3]:
# 2) Backtest: prod_hold_asym (25-70), prod_hold (20-80), champion+sniper filter

def backtest_prod_hold_asym(proba, ret, session_ids, thr_hi=0.70, thr_lo=0.25, commission_rt=COMMISSION_RT):
    pred = np.where(proba >= thr_hi, 1, np.where(proba <= thr_lo, 0, -1))
    n = len(proba)
    pos = np.zeros(n, dtype=np.float64)
    prev = 0.0
    for i in range(n):
        if session_ids is not None and i > 0 and session_ids[i] != session_ids[i - 1]:
            prev = 0.0
        if pred[i] == 1: pos[i], prev = 1.0, 1.0
        elif pred[i] == 0: pos[i], prev = -1.0, -1.0
        else: pos[i] = prev
    pos_prev = np.roll(pos, 1); pos_prev[0] = 0.0
    sess_chg = np.zeros(n, dtype=bool); sess_chg[1:] = session_ids[1:] != session_ids[:-1]
    pos_prev = np.where(sess_chg, 0.0, pos_prev)
    pos_changed = (pos != pos_prev) & ((pos != 0) | (pos_prev != 0))
    is_flip = pos_changed & (pos != 0) & (pos_prev != 0) & (pos * pos_prev < 0)
    fee_total = np.where(pos_changed, np.where(is_flip, commission_rt, commission_rt / 2.0), 0.0).sum()
    pnl_net = (pos * ret).sum() - fee_total
    trades = int(pos_changed.sum())
    return {'net_%': float(pnl_net * 100), 'trades': trades, 'avg_%_per_trade': float((pnl_net * 100) / trades) if trades > 0 else np.nan}


def backtest_prod_hold(proba, ret, session_ids, threshold=0.80, commission_rt=COMMISSION_RT):
    pred = np.where(proba >= threshold, 1, np.where(proba <= 1 - threshold, 0, -1))
    n = len(proba)
    pos = np.zeros(n, dtype=np.float64)
    prev = 0.0
    for i in range(n):
        if session_ids is not None and i > 0 and session_ids[i] != session_ids[i - 1]:
            prev = 0.0
        if pred[i] == 1: pos[i], prev = 1.0, 1.0
        elif pred[i] == 0: pos[i], prev = -1.0, -1.0
        else: pos[i] = prev
    pos_prev = np.roll(pos, 1); pos_prev[0] = 0.0
    sess_chg = np.zeros(n, dtype=bool); sess_chg[1:] = session_ids[1:] != session_ids[:-1]
    pos_prev = np.where(sess_chg, 0.0, pos_prev)
    pos_changed = (pos != pos_prev) & ((pos != 0) | (pos_prev != 0))
    is_flip = pos_changed & (pos != 0) & (pos_prev != 0) & (pos * pos_prev < 0)
    fee_total = np.where(pos_changed, np.where(is_flip, commission_rt, commission_rt / 2.0), 0.0).sum()
    pnl_net = (pos * ret).sum() - fee_total
    trades = int(pos_changed.sum())
    return {'net_%': float(pnl_net * 100), 'trades': trades, 'avg_%_per_trade': float((pnl_net * 100) / trades) if trades > 0 else np.nan}


def backtest_champion_sniper_filter(proba_champ, proba_sniper, ret, session_ids,
                                    champ_thr_hi=0.70, champ_thr_lo=0.25,
                                    sniper_thr=0.80, commission_rt=COMMISSION_RT):
    """Торгуем только когда чемпион И снайпер согласны (оба BUY или оба SELL)."""
    sig_champ = np.where(proba_champ >= champ_thr_hi, 1, np.where(proba_champ <= champ_thr_lo, 0, -1))
    sig_sniper = np.where(proba_sniper >= sniper_thr, 1, np.where(proba_sniper <= 1 - sniper_thr, 0, -1))
    # Согласованный сигнал: BUY только когда оба BUY, SELL только когда оба SELL
    agree_buy = (sig_champ == 1) & (sig_sniper == 1)
    agree_sell = (sig_champ == 0) & (sig_sniper == 0)
    pred = np.where(agree_buy, 1, np.where(agree_sell, 0, -1))
    n = len(proba_champ)
    pos = np.zeros(n, dtype=np.float64)
    prev = 0.0
    for i in range(n):
        if session_ids is not None and i > 0 and session_ids[i] != session_ids[i - 1]:
            prev = 0.0
        if pred[i] == 1: pos[i], prev = 1.0, 1.0
        elif pred[i] == 0: pos[i], prev = -1.0, -1.0
        else: pos[i] = prev
    pos_prev = np.roll(pos, 1); pos_prev[0] = 0.0
    sess_chg = np.zeros(n, dtype=bool); sess_chg[1:] = session_ids[1:] != session_ids[:-1]
    pos_prev = np.where(sess_chg, 0.0, pos_prev)
    pos_changed = (pos != pos_prev) & ((pos != 0) | (pos_prev != 0))
    is_flip = pos_changed & (pos != 0) & (pos_prev != 0) & (pos * pos_prev < 0)
    fee_total = np.where(pos_changed, np.where(is_flip, commission_rt, commission_rt / 2.0), 0.0).sum()
    pnl_net = (pos * ret).sum() - fee_total
    trades = int(pos_changed.sum())
    return {'net_%': float(pnl_net * 100), 'trades': trades, 'avg_%_per_trade': float((pnl_net * 100) / trades) if trades > 0 else np.nan}

print('Backtest functions ready.')

Backtest functions ready.


In [4]:
# 3) Обучение: champion (cat) + snipers (cat, logreg, rf)

scaler = StandardScaler()
X_tr = scaler.fit_transform(tr[FEATURES_NEW].fillna(0))
X_va = scaler.transform(va[FEATURES_NEW].fillna(0))
X_te_f = scaler.transform(te_full[FEATURES_NEW].fillna(0))
X_va_f = scaler.transform(val_full[FEATURES_NEW].fillna(0))
y_tr = tr['y'].values

proba_te = {}
proba_val = {}

# Champion: CatBoost
if HAS_CAT:
    cat_model = cb.CatBoostClassifier(iterations=300, depth=6, learning_rate=0.05, random_seed=42, verbose=0)
    cat_model.fit(X_tr, y_tr)
    proba_te['cat'] = cat_model.predict_proba(X_te_f)[:, 1]
    proba_val['cat'] = cat_model.predict_proba(X_va_f)[:, 1]

# Snipers: LogReg, RF, (cat уже есть)
logreg = LogisticRegression(max_iter=1200, random_state=42)
logreg.fit(X_tr, y_tr)
proba_te['logreg'] = logreg.predict_proba(X_te_f)[:, 1]
proba_val['logreg'] = logreg.predict_proba(X_va_f)[:, 1]

rf = RandomForestClassifier(n_estimators=300, max_depth=14, min_samples_split=8, min_samples_leaf=4, max_features='sqrt', random_state=42)
rf.fit(X_tr, y_tr)
proba_te['rf'] = rf.predict_proba(X_te_f)[:, 1]
proba_val['rf'] = rf.predict_proba(X_va_f)[:, 1]

ret = te_full['ret_next'].values
sess = te_full['session_key'].values
ret_val = val_full['ret_next'].values
sess_val = val_full['session_key'].values

print('Models trained: cat, logreg, rf')
print('te_full rows:', len(ret), '| val_full:', len(ret_val))

Models trained: cat, logreg, rf
te_full rows: 74437 | val_full: 45592


In [5]:
# 4) Базовые метрики: champion solo, снайперы solo

results = []

if HAS_CAT:
    bt_champ = backtest_prod_hold_asym(proba_te['cat'], ret, sess)
    results.append({'variant': 'champion_solo', 'champ': 'cat', 'sniper': '-', 'net_%': bt_champ['net_%'], 'trades': bt_champ['trades'], 'avg_%': bt_champ['avg_%_per_trade']})
    print('Champion (cat 25-70) solo:', f"net={bt_champ['net_%']:.1f}%", f"trades={bt_champ['trades']}", f"avg={bt_champ['avg_%_per_trade']:.4f}%")
    print()

print('Snipers (20-80) solo:')
for name in ['cat', 'logreg', 'rf']:
    if name not in proba_te:
        continue
    bt = backtest_prod_hold(proba_te[name], ret, sess, threshold=0.80)
    results.append({'variant': 'sniper_solo', 'champ': '-', 'sniper': name, 'net_%': bt['net_%'], 'trades': bt['trades'], 'avg_%': bt['avg_%_per_trade']})
    print(f'  {name}: net={bt["net_%"]:.1f}% trades={bt["trades"]} avg={bt["avg_%_per_trade"]:.4f}%')

Champion (cat 25-70) solo: net=1503.6% trades=768 avg=1.9579%

Snipers (20-80) solo:
  cat: net=655.1% trades=272 avg=2.4085%
  logreg: net=370.4% trades=86 avg=4.3065%
  rf: net=252.0% trades=92 avg=2.7392%


In [6]:
# 5) Champion + Sniper filter: cat (champ) фильтруем каждым снайпером

if HAS_CAT:
    print('Champion (cat 25-70) + Sniper filter (20-80):')
    print('-' * 60)
    for sniper_name in ['cat', 'logreg', 'rf']:
        if sniper_name not in proba_te:
            continue
        bt = backtest_champion_sniper_filter(proba_te['cat'], proba_te[sniper_name], ret, sess,
                                            champ_thr_hi=0.70, champ_thr_lo=0.25, sniper_thr=0.80)
        results.append({'variant': 'champ+sniper', 'champ': 'cat', 'sniper': sniper_name, 'net_%': bt['net_%'], 'trades': bt['trades'], 'avg_%': bt['avg_%_per_trade']})
        print(f"  cat + {sniper_name}: net={bt['net_%']:.1f}% trades={bt['trades']} avg={bt['avg_%_per_trade']:.4f}%")
    print()
else:
    print('CatBoost not available, skip champion.')

Champion (cat 25-70) + Sniper filter (20-80):
------------------------------------------------------------
  cat + cat: net=655.1% trades=272 avg=2.4085%
  cat + logreg: net=251.2% trades=53 avg=4.7402%
  cat + rf: net=247.6% trades=92 avg=2.6916%



In [7]:
# 6) Сводная таблица и выводы

res_df = pd.DataFrame(results)
print('=== Сводка (test full) ===')
print(res_df.to_string(index=False))

if HAS_CAT:
    champ_solo = res_df[res_df['variant'] == 'champion_solo'].iloc[0]
    combos = res_df[res_df['variant'] == 'champ+sniper']
    best_combo = combos.loc[combos['avg_%'].idxmax()] if len(combos) > 0 else None
    print()
    print('Выводы:')
    print(f"  Champion solo: {champ_solo['net_%']:.1f}% net, {champ_solo['trades']} trades, {champ_solo['avg_%']:.4f}% avg")
    if best_combo is not None:
        print(f"  Лучший champ+sniper: {best_combo['champ']}+{best_combo['sniper']} → net={best_combo['net_%']:.1f}%, trades={best_combo['trades']}, avg={best_combo['avg_%']:.4f}%")
        if best_combo['avg_%'] > champ_solo['avg_%']:
            print('  → Фильтр снайпером дал прирост avg/trade.')
        else:
            print('  → Champion solo сильнее по avg или trades.')

=== Сводка (test full) ===
      variant champ sniper       net_%  trades    avg_%
champion_solo   cat      - 1503.646025     768 1.957872
  sniper_solo     -    cat  655.104836     272 2.408474
  sniper_solo     - logreg  370.355445      86 4.306459
  sniper_solo     -     rf  252.010699      92 2.739247
 champ+sniper   cat    cat  655.104836     272 2.408474
 champ+sniper   cat logreg  251.229029      53 4.740170
 champ+sniper   cat     rf  247.629914      92 2.691630

Выводы:
  Champion solo: 1503.6% net, 768 trades, 1.9579% avg
  Лучший champ+sniper: cat+logreg → net=251.2%, trades=53, avg=4.7402%
  → Фильтр снайпером дал прирост avg/trade.
